In [11]:
import os
import sys

PROJECT_DIR = os.path.dirname(os.getcwd())

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

In [12]:
import copy

import torch
import torch.nn as nn
import yaml
from torch.utils.data import DataLoader
from torchmetrics.text import WordErrorRate
from tqdm.auto import tqdm
from transformers.trainer_pt_utils import LengthGroupedSampler

from srcs.datasets.vicocktail import Collator, load_vicocktail
from srcs.nets.backend.encoder.conformer_encoder import ConformerEncoder
from srcs.nets.backend.nets_utils import make_non_pad_mask
from srcs.nets.e2e import get_model
from srcs.nets.utils import ctc_decode, parameter_count
from srcs.spm.text_transofm import TextTransform

with open(os.path.join(PROJECT_DIR, "config.yaml"), encoding="utf-8") as file:
    configs = yaml.safe_load(file)

SEED = 42
TRAIN_FRACTION = 0.1
VALIDATION_FRACTION = 1.0
EPOCHS = 10
PATIENCE = 3
BATCH_SIZE = 16
NUM_WORKERS = 0
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.005
REFINER_BLOCKS = 1
BASELINE_CHECKPOINT = os.path.join(
    PROJECT_DIR, "checkpoints", "baseline", "checkpoint-305877"
)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
device

device(type='cuda')

In [13]:
dataset = load_vicocktail(
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    splits=("train", "val"),
    seed=SEED,
)
text_transform = TextTransform()
train_collator = Collator(text_transform, "train")
validation_collator = Collator(text_transform, "val")

train_lengths = [int(value) for value in dataset["train"]["video_length"]]
train_sampler = LengthGroupedSampler(BATCH_SIZE, lengths=train_lengths)
train_loader = DataLoader(
    dataset["train"],
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    collate_fn=train_collator,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
)
validation_loader = DataLoader(
    dataset["val"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=validation_collator,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
)

baseline = get_model(
    "baseline",
    text_transform.vocab_size,
    checkpoint_path=BASELINE_CHECKPOINT,
    **configs["model"],
).to(device)
baseline.requires_grad_(False).eval()

print(f"Train samples: {len(dataset['train']):,}")
print(f"Validation samples: {len(dataset['val']):,}")
print(f"Baseline parameters: {parameter_count(baseline):,}")

Train samples: 18,823
Validation samples: 5,844
Baseline parameters: 24,798,074


In [ ]:
class VisualRefiner(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=256,
        heads=4,
        linear_units=512,
        num_blocks=1,
        dropout=0.1,
        use_visual=False,
    ):
        super().__init__()
        
        self.

In [4]:
class ResidualCTCRefiner(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=256,
        heads=4,
        linear_units=512,
        num_blocks=1,
        dropout=0.1,
        use_visual=False,
    ):
        super().__init__()
        self.use_visual = use_visual
        self.posterior_proj = nn.Sequential(
            nn.Linear(vocab_size, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        if use_visual:
            self.visual_norm = nn.LayerNorm(dim)
            self.h2_norm = nn.LayerNorm(dim)
            self.fuse = nn.Sequential(
                nn.Linear(dim * 3, dim),
                nn.LayerNorm(dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )

        self.encoder = ConformerEncoder(
            attention_dim=dim,
            attention_heads=heads,
            linear_units=linear_units,
            num_blocks=num_blocks,
            dropout_rate=dropout,
            positional_dropout_rate=dropout,
            attention_dropout_rate=0.0,
            cnn_module_kernel=31,
        )
        self.delta_head = nn.Linear(dim, vocab_size)
        nn.init.zeros_(self.delta_head.weight)
        nn.init.zeros_(self.delta_head.bias)

    def forward(self, posterior, mask, f_visual=None, h2=None):
        hidden = self.posterior_proj(posterior)

        if self.use_visual:
            if f_visual is None or h2 is None:
                raise ValueError("Visual contexts are required.")

            hidden = self.fuse(
                torch.cat(
                    [hidden, self.visual_norm(f_visual), self.h2_norm(h2)],
                    dim=-1,
                )
            )

        hidden = self.encoder(hidden, mask)[0]
        return self.delta_head(hidden)

In [5]:
refiners = {
    "posterior": ResidualCTCRefiner(
        text_transform.vocab_size,
        dim=configs["model"]["attention_dim"],
        heads=configs["model"]["attention_heads"],
        num_blocks=REFINER_BLOCKS,
        use_visual=False,
    ).to(device),
    "visual": ResidualCTCRefiner(
        text_transform.vocab_size,
        dim=configs["model"]["attention_dim"],
        heads=configs["model"]["attention_heads"],
        num_blocks=REFINER_BLOCKS,
        use_visual=True,
    ).to(device),
}
optimizers = {
    name: torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    for name, model in refiners.items()
}
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

for name, refiner in refiners.items():
    print(f"{name} parameters: {parameter_count(refiner):,}")

posterior parameters: 2,604,986
visual parameters: 2,803,386


In [6]:
def move_batch(batch):
    return {
        name: value.to(device, non_blocking=True) if torch.is_tensor(value) else value
        for name, value in batch.items()
    }


@torch.no_grad()
def baseline_pass(batch):
    with torch.autocast(
        device_type=device.type, dtype=torch.float16, enabled=use_amp
    ):
        contexts = baseline.get_contexts(
            batch["videos"], batch["video_lengths"]
        )
        _, logits = baseline.ctc(
            contexts["encoded_features"], contexts["input_lengths"]
        )
        posterior = torch.softmax(logits, dim=-1)

    return {
        "logits": logits.detach(),
        "posterior": posterior.detach(),
        "f_visual": contexts["f_visual"].detach(),
        "h2": contexts["h2"].detach(),
        "input_lengths": contexts["input_lengths"],
    }


def loss_from_logits(logits, input_lengths, labels, label_lengths):
    labels = labels.to(dtype=torch.long)
    label_lengths = label_lengths.to(dtype=torch.long)
    positions = torch.arange(labels.size(1), device=labels.device)
    label_mask = positions.unsqueeze(0) < label_lengths.unsqueeze(1)
    targets = labels.masked_select(label_mask)
    log_probs = logits.float().log_softmax(dim=-1).transpose(0, 1)
    loss = baseline.ctc.ctc_loss(
        log_probs, targets, input_lengths, label_lengths
    )
    return loss / logits.size(0)


def decode_logits(logits, input_lengths):
    token_ids = ctc_decode(logits, input_lengths, text_transform.blank_id)
    return [text_transform.decode(ids) for ids in token_ids]


def decode_references(labels, label_lengths):
    return [
        text_transform.decode(label[: int(length)])
        for label, length in zip(labels, label_lengths)
    ]


def make_mask(input_lengths):
    return make_non_pad_mask(input_lengths).to(device).unsqueeze(-2)

In [7]:
batch = move_batch(next(iter(train_loader)))
first_pass = baseline_pass(batch)
mask = make_mask(first_pass["input_lengths"])

with torch.no_grad(), torch.autocast(
    device_type=device.type, dtype=torch.float16, enabled=use_amp
):
    posterior_delta = refiners["posterior"](
        first_pass["posterior"], mask
    )
    visual_delta = refiners["visual"](
        first_pass["posterior"],
        mask,
        first_pass["f_visual"],
        first_pass["h2"],
    )

print("Pg:", tuple(first_pass["posterior"].shape))
print("H_visual:", tuple(first_pass["f_visual"].shape))
print("H2:", tuple(first_pass["h2"].shape))
print("R(Pg) initial max delta:", posterior_delta.abs().max().item())
print("R(Pg, H_visual, H2) initial max delta:", visual_delta.abs().max().item())

Pg: (16, 375, 3002)
H_visual: (16, 375, 256)
H2: (16, 375, 256)
R(Pg) initial max delta: 0.0
R(Pg, H_visual, H2) initial max delta: 0.0


In [8]:
def train_epoch(active):
    for refiner in refiners.values():
        refiner.train()

    totals = {name: 0.0 for name in refiners}
    steps = {name: 0 for name in refiners}
    progress = tqdm(train_loader, desc="train")

    for raw_batch in progress:
        batch = move_batch(raw_batch)
        first_pass = baseline_pass(batch)
        mask = make_mask(first_pass["input_lengths"])

        for name, refiner in refiners.items():
            if not active[name]:
                continue

            optimizers[name].zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device.type, dtype=torch.float16, enabled=use_amp
            ):
                if name == "posterior":
                    delta = refiner(first_pass["posterior"], mask)
                else:
                    delta = refiner(
                        first_pass["posterior"],
                        mask,
                        first_pass["f_visual"],
                        first_pass["h2"],
                    )

                logits = first_pass["logits"] + delta
                loss = loss_from_logits(
                    logits,
                    first_pass["input_lengths"],
                    batch["labels"],
                    batch["label_lengths"],
                )

            scaler.scale(loss).backward()
            scaler.step(optimizers[name])
            totals[name] += loss.detach().item()
            steps[name] += 1

        scaler.update()
        progress.set_postfix(
            {
                name: totals[name] / max(steps[name], 1)
                for name in refiners
                if steps[name]
            }
        )

    return {name: totals[name] / max(steps[name], 1) for name in refiners}


@torch.no_grad()
def evaluate(include_shuffle=False):
    for refiner in refiners.values():
        refiner.eval()

    names = ["baseline", "posterior", "visual"]

    if include_shuffle:
        names.append("visual_shuffle")

    metrics = {name: WordErrorRate() for name in names}

    for raw_batch in tqdm(validation_loader, desc="validation"):
        batch = move_batch(raw_batch)
        first_pass = baseline_pass(batch)
        mask = make_mask(first_pass["input_lengths"])

        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=use_amp
        ):
            posterior_logits = first_pass["logits"] + refiners["posterior"](
                first_pass["posterior"], mask
            )
            visual_logits = first_pass["logits"] + refiners["visual"](
                first_pass["posterior"],
                mask,
                first_pass["f_visual"],
                first_pass["h2"],
            )

            if include_shuffle:
                permutation = torch.roll(
                    torch.arange(first_pass["logits"].size(0), device=device),
                    shifts=1,
                )
                shuffled_logits = first_pass["logits"] + refiners["visual"](
                    first_pass["posterior"],
                    mask,
                    first_pass["f_visual"][permutation],
                    first_pass["h2"][permutation],
                )

        references = decode_references(
            batch["labels"], batch["label_lengths"]
        )
        metrics["baseline"].update(
            decode_logits(
                first_pass["logits"], first_pass["input_lengths"]
            ),
            references,
        )
        metrics["posterior"].update(
            decode_logits(posterior_logits, first_pass["input_lengths"]),
            references,
        )
        metrics["visual"].update(
            decode_logits(visual_logits, first_pass["input_lengths"]),
            references,
        )

        if include_shuffle:
            metrics["visual_shuffle"].update(
                decode_logits(shuffled_logits, first_pass["input_lengths"]),
                references,
            )

    return {f"{name}_wer": metric.compute().item() for name, metric in metrics.items()}

In [ ]:
best = {
    name: {"wer": float("inf"), "epoch": 0, "bad_epochs": 0, "state": None}
    for name in refiners
}
active = {name: True for name in refiners}
history = []

for epoch in range(1, EPOCHS + 1):
    train_losses = train_epoch(active)
    scores = evaluate()
    record = {"epoch": epoch, **train_losses, **scores}
    history.append(record)
    print(record)

    for name in refiners:
        score = scores[f"{name}_wer"]

        if score < best[name]["wer"]:
            best[name]["wer"] = score
            best[name]["epoch"] = epoch
            best[name]["bad_epochs"] = 0
            best[name]["state"] = copy.deepcopy(
                {key: value.detach().cpu() for key, value in refiners[name].state_dict().items()}
            )
        else:
            best[name]["bad_epochs"] += 1

            if best[name]["bad_epochs"] >= PATIENCE:
                active[name] = False

    if not any(active.values()):
        break

for name, refiner in refiners.items():
    refiner.load_state_dict(best[name]["state"])
    print(f"Best {name}: epoch={best[name]['epoch']}, WER={best[name]['wer']:.6f}")

train:  34%|███▍      | 405/1177 [08:08<25:33,  1.99s/it, posterior=45.9, visual=45.9]

In [ ]:
def length_matched_permutation(lengths):
    count = lengths.numel()
    permutation = torch.arange(count, device=lengths.device)

    if count < 2:
        return permutation

    order = torch.argsort(lengths)
    pair_limit = count - count % 2
    left = order[:pair_limit:2]
    right = order[1:pair_limit:2]
    permutation[left] = right
    permutation[right] = left
    return permutation


@torch.no_grad()
def evaluate_probes():
    for refiner in refiners.values():
        refiner.eval()

    names = [
        "baseline",
        "posterior",
        "visual",
        "visual_zero",
        "visual_shuffle_hv",
        "visual_shuffle_h2",
        "visual_shuffle_both",
    ]
    metrics = {name: WordErrorRate() for name in names}
    length_gap_sum = 0
    length_gap_count = 0
    length_gap_max = 0
    shuffled_sample_count = 0
    total_sample_count = 0

    for raw_batch in tqdm(validation_loader, desc="probe"):
        batch = move_batch(raw_batch)
        first_pass = baseline_pass(batch)
        input_lengths = first_pass["input_lengths"]
        mask = make_mask(input_lengths)
        permutation = length_matched_permutation(input_lengths)
        gaps = (input_lengths - input_lengths[permutation]).abs()
        length_gap_sum += gaps.sum().item()
        length_gap_count += gaps.numel()
        length_gap_max = max(length_gap_max, gaps.max().item())
        shuffled_sample_count += permutation.ne(
            torch.arange(permutation.numel(), device=device)
        ).sum().item()
        total_sample_count += permutation.numel()

        posterior = first_pass["posterior"]
        f_visual = first_pass["f_visual"]
        h2 = first_pass["h2"]
        base_logits = first_pass["logits"]
        zero_visual = torch.zeros_like(f_visual)
        zero_h2 = torch.zeros_like(h2)
        shuffled_visual = f_visual[permutation]
        shuffled_h2 = h2[permutation]

        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=use_amp
        ):
            logits_by_name = {
                "baseline": base_logits,
                "posterior": base_logits
                + refiners["posterior"](posterior, mask),
                "visual": base_logits
                + refiners["visual"](posterior, mask, f_visual, h2),
                "visual_zero": base_logits
                + refiners["visual"](
                    posterior, mask, zero_visual, zero_h2
                ),
                "visual_shuffle_hv": base_logits
                + refiners["visual"](
                    posterior, mask, shuffled_visual, h2
                ),
                "visual_shuffle_h2": base_logits
                + refiners["visual"](
                    posterior, mask, f_visual, shuffled_h2
                ),
                "visual_shuffle_both": base_logits
                + refiners["visual"](
                    posterior, mask, shuffled_visual, shuffled_h2
                ),
            }

        references = decode_references(
            batch["labels"], batch["label_lengths"]
        )

        for name, logits in logits_by_name.items():
            metrics[name].update(
                decode_logits(logits, input_lengths), references
            )

    scores = {
        f"{name}_wer": metric.compute().item()
        for name, metric in metrics.items()
    }
    scores["shuffle_mean_length_gap"] = length_gap_sum / length_gap_count
    scores["shuffle_max_length_gap"] = length_gap_max
    scores["shuffle_fraction"] = shuffled_sample_count / total_sample_count
    return scores


final_scores = evaluate_probes()
comparison = {
    "Baseline": final_scores["baseline_wer"],
    "R(Pg)": final_scores["posterior_wer"],
    "Normal": final_scores["visual_wer"],
    "Zero Hv and H2": final_scores["visual_zero_wer"],
    "Shuffle Hv only": final_scores["visual_shuffle_hv_wer"],
    "Shuffle H2 only": final_scores["visual_shuffle_h2_wer"],
    "Shuffle Hv and H2": final_scores["visual_shuffle_both_wer"],
}
normal_wer = final_scores["visual_wer"]
damage = {
    "Zero Hv and H2": final_scores["visual_zero_wer"] - normal_wer,
    "Shuffle Hv only": final_scores["visual_shuffle_hv_wer"] - normal_wer,
    "Shuffle H2 only": final_scores["visual_shuffle_h2_wer"] - normal_wer,
    "Shuffle Hv and H2": final_scores["visual_shuffle_both_wer"] - normal_wer,
}

print(comparison)
print("WER increase relative to normal:", damage)
print(
    "Length-matched shuffle gap:",
    {
        "mean_frames": final_scores["shuffle_mean_length_gap"],
        "max_frames": final_scores["shuffle_max_length_gap"],
        "fraction": final_scores["shuffle_fraction"],
    },
)